In [1]:
# !pip install -q ultralytics

In [2]:
from ultralytics import YOLO

model  = YOLO("yolo11s.pt")

print(model)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

In [3]:
# import cv2

# cap = cv2.VideoCapture("../../Datasets_for_my_practice/poeple_detection/people-walking.mp4")


In [4]:
# while True:

#     ret, frame = cap.read()

#     if not ret:
#         break

#     results = model.predict(
#         source=frame,
#         classes=[0],
#         conf=0.25,
#         verbose=False
#     )

#     annotated_frame = results[0].plot()

#     cv2.imshow("People Detection", annotated_frame)

#     key = cv2.waitKey(1) & 0xFF

#     # Press q to quit
#     if key == ord("q"):
#         break

#     # Close window using X
#     if cv2.getWindowProperty("People Detection", cv2.WND_PROP_VISIBLE) < 1:
#         break

# cap.release()
# cv2.destroyAllWindows()

In [14]:
import cv2
import numpy as np

cap = cv2.VideoCapture("../../Datasets_for_my_practice/poeple_detection/people-walking.mp4")

model  = YOLO("yolo11s.pt")

cv2.namedWindow("People Detection", cv2.WINDOW_NORMAL)
cv2.resizeWindow("People Detection", 1200, 700)

# LINE1_Y = 700*0.2
# LINE2_Y = 700*0.8

in_count = 0
out_count = 0
counted_ids = set()

previous_positions = {}

heatmap = None

ret, frame = cap.read()

if not ret:
    raise RuntimeError("Could not read the first frame from the video.")

height, width = frame.shape[:2]

fps = int(cap.get(cv2.CAP_PROP_FPS))

video_writer = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

height, width = frame.shape[:2]
LINE1_Y = int(height * 0.3)
LINE2_Y = int(height * 0.7)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while True:

    ret, frame = cap.read()
    

    if not ret:
        break
    

    
    if heatmap is None:
        heatmap = np.zeros((height, width), dtype=np.float32)

    results = model.track(
        source=frame,
        classes=[0],
        conf=0.10,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )
    
    boxes = results[0].boxes
    # print(boxes.id)
    
    for box in boxes:

        track_id = int(box.id)

        class_id = int(box.cls)

        confidence = float(box.conf)

        # print(track_id, class_id, confidence)
        
        
        x1, y1, x2, y2 = box.xyxy[0]

        center_x = int((x1 + x2) / 2)
        center_y = int((y1 + y2) / 2)
        
        cv2.circle(
            heatmap,
            (center_x, center_y),
            1,
            1,
            -1
        )

        # print(center_x, center_y)
        
        if track_id in previous_positions:
            
            previous_center = previous_positions[track_id]
            
            previous_x, previous_y = previous_center
            
            # print(previous_center)
            
            # if previous_y < center_y:
            #     print(f"ID {track_id} Moving Down")

            # elif previous_y > center_y:
            #     print(f"ID {track_id} Moving Up")
                
            # Moving Down -> IN
            if (
                previous_y < LINE1_Y <= center_y
                and track_id not in counted_ids
            ):
                in_count += 1
                counted_ids.add(track_id)

            # Moving Up -> OUT
            elif (
                previous_y > LINE2_Y >= center_y
                and track_id not in counted_ids
            ):
                out_count += 1
                counted_ids.add(track_id)
                
        
        previous_positions[track_id] = (center_x, center_y)
        
        # print(previous_positions)
    

    annotated_frame = results[0].plot()
    
    cv2.line(
    annotated_frame,
    (0, LINE1_Y),
    (frame.shape[1], LINE1_Y),
    (0,255,0),
    2
    )

    cv2.line(
        annotated_frame,
        (0, LINE2_Y),
        (frame.shape[1], LINE2_Y),
        (255,0,0),
        2
    )
    
    cv2.putText(
    annotated_frame,
    f"IN : {in_count}",
    (20, 40),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (0,255,0),
    2
    )

    cv2.putText(
        annotated_frame,
        f"OUT : {out_count}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,0,0),
        2
    )
    
    video_writer.write(annotated_frame)

    cv2.imshow("People Detection", annotated_frame)

    key = cv2.waitKey(1) & 0xFF

    # Press q to quit
    if key == ord("q"):
        break

    # Close window using X
    if cv2.getWindowProperty("People Detection", cv2.WND_PROP_VISIBLE) < 1:
        break

heatmap = cv2.GaussianBlur(
    heatmap,
    (51,51),
    0
)

heatmap = cv2.normalize(
    heatmap,
    None,
    0,
    255,
    cv2.NORM_MINMAX
)

heatmap = heatmap.astype(np.uint8)

heatmap_color = cv2.applyColorMap(
    heatmap,
    cv2.COLORMAP_JET
)

cv2.imwrite("heatmap.jpg", heatmap_color)


video_writer.release()
cap.release()
cv2.destroyAllWindows()